<a href="https://colab.research.google.com/github/ThinkingBeyond/BeyondAI-2025/blob/main/Aryan%20Basnet%2C%20Arnav%20Maharjan%20and%20Ashila%20A%20M%20Ardiyansyah/05_dataset5_HIC_pneumonia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **NOTE ON RUNTIME AND OUTPUTS**

# Google Colab may have disconnected or reset the runtime during long training sessions, crashes, or memory interruptions. When this occurred, some previously displayed outputs in the notebook were no longer visible. However, all results remained saved and logged correctly. Each model’s complete metrics and metadata were stored as JSON files in my Google Drive folder:

# [https://drive.google.com/drive/folders/1ejlJaZhHEBm-1khLBJ--mbG2pg5TZHoJ?usp=sharing](https://drive.google.com/drive/folders/1ejlJaZhHEBm-1khLBJ--mbG2pg5TZHoJ?usp=sharing)

# These JSON files contain the full and reliable outputs for all models across all datasets, even if certain notebook outputs were lost due to runtime resets.

# ==============================
# SETUP: Freeze all package versions
# ==============================
Ensure reproducibility by installing the exact versions of packages used in these notebooks. This includes pre-installed packages in Colab.

The packages and versions used are:

- numpy==1.25.2
- pandas==2.1.1
- matplotlib==3.8.0
- seaborn==0.12.2
- scikit-learn==1.3.2
- tensorflow==2.15.0
- keras==2.15.0
- scipy==1.11.2
- opencv-python==4.9.0.73
- Pillow==10.0.1
- h5py==3.9.0
- google-colab==2.0.0

In [ ]:
# ==========================================
# CHEST X-RAY CLASSIFICATION - HIC DATASET
# ==========================================

# STEP 1: METADATA
DATASET_NAME = "dataset_chestxray2017"
COUNTRY_INCOME_LEVEL = "HIC"  # High-Income Country
NUM_CLASSES = 2  # Normal vs Pneumonia
CLASS_NAMES = ['NORMAL', 'PNEUMONIA']

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 20

# STEP 2: MOUNT DRIVE (if using Colab)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/xray_research_results

### ADDED: Persistent dataset folder on Drive
PERSISTENT_DATA_DIR = "/content/drive/MyDrive/xray_datasets"
!mkdir -p $PERSISTENT_DATA_DIR

### ADDED: Auto-copy dataset ONCE only
import os
if not os.path.exists(f"{PERSISTENT_DATA_DIR}/{DATASET_NAME}"):
    print("Copying dataset to Drive for permanent storage...")
    !cp -r /content/dataset_chestxray2017 $PERSISTENT_DATA_DIR/
else:
    print("Dataset already stored in Drive. Using persistent copy.")

# STEP 3: IMPORT LIBRARIES (unchanged)
import os, gc, json, time
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import f1_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# STEP 4: DATA PATHS

### ADDED: Use persistent dataset path, fallback if missing
DATASET_PATH = f"{PERSISTENT_DATA_DIR}/{DATASET_NAME}/chest_xray"
if not os.path.exists(DATASET_PATH):
    DATASET_PATH = "/content/dataset_chestxray2017/chest_xray"

train_dir = os.path.join(DATASET_PATH, 'train')
test_dir  = os.path.join(DATASET_PATH, 'test')

# STEP 5: DATA GENERATORS (unchanged)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.1,
    brightness_range=[0.9,1.1],
    horizontal_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1,
    validation_split=0.2
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training',
    shuffle=True
)

validation_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

print(f"\nTrain samples: {train_generator.samples}")
print(f"Val samples:   {validation_generator.samples}")
print(f"Test samples:  {test_generator.samples}")
print(f"Classes: {train_generator.class_indices}")

# STEP 6: MODEL ARCHITECTURES (unchanged)
def create_baseline_cnn(input_shape=(224,224,3), num_classes=2):
    model = keras.Sequential([
        layers.Conv2D(32, (3,3), activation='relu', input_shape=input_shape),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(64, (3,3), activation='relu'),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(64, (3,3), activation='relu'),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ])
    return model

def create_transfer_model(base_model_name, input_shape=(224,224,3), num_classes=2):
    if base_model_name == 'MobileNetV2':
        base = tf.keras.applications.MobileNetV2(input_shape=input_shape, include_top=False, weights='imagenet')
    elif base_model_name == 'EfficientNetB0':
        base = tf.keras.applications.EfficientNetB0(input_shape=input_shape, include_top=False, weights='imagenet')
    elif base_model_name == 'ResNet50':
        base = tf.keras.applications.ResNet50(input_shape=input_shape, include_top=False, weights='imagenet')
    else:
        raise ValueError(f"Unknown model: {base_model_name}")

    base.trainable = False
    inputs = keras.Input(shape=input_shape)
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    model = keras.Model(inputs, outputs)
    return model

# STEP 7: TRAINING AND EVALUATION FUNCTION

def train_and_evaluate(model, model_name, train_gen, val_gen, test_gen):
    print(f"\n{'='*50}\nTraining {model_name}\n{'='*50}")
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    ### ADDED: Checkpoint path
    ckpt_dir = f"/content/drive/MyDrive/xray_checkpoints/{DATASET_NAME}/{model_name}"
    os.makedirs(ckpt_dir, exist_ok=True)
    ckpt_path = f"{ckpt_dir}/epoch_{{epoch:02d}}.weights.h5"

    ### ADDED: ModelCheckpoint callback
    checkpoint_cb = keras.callbacks.ModelCheckpoint(
        filepath=ckpt_path,
        save_weights_only=True,
        save_freq='epoch',
        verbose=1
    )

    early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

    ### ADDED: Auto-resume if checkpoint exists
    latest_ckpt = sorted(os.listdir(ckpt_dir))[-1] if os.listdir(ckpt_dir) else None
    if latest_ckpt:
        print(f"Resuming from checkpoint: {latest_ckpt}")
        model.load_weights(os.path.join(ckpt_dir, latest_ckpt))

    start_time = time.time()
    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=EPOCHS,
        callbacks=[early_stop, checkpoint_cb],
        verbose=1
    )
    training_time = (time.time() - start_time)/60

    # Evaluate test
    test_loss, test_acc = model.evaluate(test_gen, verbose=0)
    predictions = model.predict(test_gen)
    y_pred = (predictions > 0.5).astype(int).flatten()
    y_true = test_gen.classes

    f1_per_class = [
        f1_score(y_true==0, y_pred==0),
        f1_score(y_true==1, y_pred==1)
    ]
    f1_weighted = f1_score(y_true, y_pred, average='weighted')
    cm = confusion_matrix(y_true, y_pred)

    results = {
        'dataset_name': DATASET_NAME,
        'country_income': COUNTRY_INCOME_LEVEL,
        'model_name': model_name,
        'num_classes': NUM_CLASSES,
        'class_names': CLASS_NAMES,
        'f1_per_class': f1_per_class,
        'f1_weighted': float(f1_weighted),
        'confusion_matrix': cm.tolist(),
        'training_time_minutes': float(training_time),
        'num_images_train': train_gen.samples,
        'num_images_val': val_gen.samples,
        'num_images_test': test_gen.samples,
        'num_parameters': int(model.count_params()),
        'test_accuracy': float(test_acc),
        'epochs_trained': len(history.history['loss'])
    }

    filename = f'/content/drive/MyDrive/xray_research_results/{DATASET_NAME}_{model_name}_results.json'
    with open(filename, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"Results saved to: {filename}")

    del model
    tf.keras.backend.clear_session()
    gc.collect()

    return results

# STEP 8: TRAIN ALL MODELS (unchanged)
all_results = []

model = create_baseline_cnn()
all_results.append(train_and_evaluate(model, 'BaselineCNN', train_generator, validation_generator, test_generator))

model = create_transfer_model('MobileNetV2')
all_results.append(train_and_evaluate(model, 'MobileNetV2', train_generator, validation_generator, test_generator))

model = create_transfer_model('EfficientNetB0')
all_results.append(train_and_evaluate(model, 'EfficientNetB0', train_generator, validation_generator, test_generator))

model = create_transfer_model('ResNet50')
all_results.append(train_and_evaluate(model, 'ResNet50', train_generator, validation_generator, test_generator))

# STEP 9: SUMMARY (unchanged)
print("\n" + "="*60)
print("TRAINING COMPLETE - SUMMARY")
print("="*60)
for result in all_results:
    print(f"\n{result['model_name']}:")
    print(f"  Weighted F1: {result['f1_weighted']:.4f}")
    print(f"  Test Accuracy: {result['test_accuracy']:.4f}")
    print(f"  Training Time: {result['training_time_minutes']:.2f} min")
    print(f"  Parameters: {result['num_parameters']:,}")
print(f"\nAll results saved to: /content/drive/MyDrive/xray_research_results/")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found 4187 images belonging to 2 classes.
Found 1045 images belonging to 2 classes.
Found 624 images belonging to 2 classes.

Train samples: 4187
Val samples:   1045
Test samples:  624
Classes: {'NORMAL': 0, 'PNEUMONIA': 1}

Training BaselineCNN
Epoch 1/20
131/131 ━━━━━━━━━━━━━━━━━━━━ 591s 4s/step - accuracy: 0.7623 - loss: 0.6139 - val_accuracy: 0.8612 - val_loss: 0.3009
Epoch 2/20
131/131 ━━━━━━━━━━━━━━━━━━━━ 591s 4s/step - accuracy: 0.8556 - loss: 0.3240 - val_accuracy: 0.8775 - val_loss: 0.2836
Epoch 3/20
131/131 ━━━━━━━━━━━━━━━━━━━━ 585s 4s/step - accuracy: 0.8755 - loss: 0.2853 - val_accuracy: 0.8507 - val_loss: 0.3033
Epoch 4/20
131/131 ━━━━━━━━━━━━━━━━━━━━ 577s 4s/step - accuracy: 0.8854 - loss: 0.2674 - val_accuracy: 0.8890 - val_loss: 0.2454
Epoch 5/20
131/131 ━━━━━━━━━━━━━━━━━━━━ 580s 4s/step - accuracy: 0.8919 - loss: 0.2378 - val_accuracy: 0.8976

In [ ]:
import zipfile
import os

zip_path = '/content/drive/MyDrive/ChestXRay2017.zip'
extract_path = '/content/drive/MyDrive/xray_datasets/dataset_chestxray2017'

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [ ]:
train_classes = [cls for cls in os.listdir(train_path) if cls != '.DS_Store']
print("Classes in train:", train_classes)

Classes in train: ['PNEUMONIA', 'NORMAL']


In [ ]:
def count_images(path):
    counts = {}
    total = 0
    for cls in os.listdir(path):
        if cls == '.DS_Store':
            continue  # skip hidden macOS file
        cls_path = os.path.join(path, cls)
        if not os.path.isdir(cls_path):
            continue
        num = len([f for f in os.listdir(cls_path) if f.lower().endswith(('.png','.jpg','.jpeg'))])
        counts[cls] = num
        total += num
    return total, counts

In [ ]:
images = [f for f in os.listdir(cls_train_path)
          if f.lower().endswith(('.png','.jpg','.jpeg'))]  # ignores .DS_Store

In [ ]:
import os
import shutil
import random

# Paths
base_path = '/content/drive/MyDrive/xray_datasets/dataset_chestxray2017/chest_xray'
train_path = os.path.join(base_path, 'train')
val_path   = os.path.join(base_path, 'val')
test_path  = os.path.join(base_path, 'test')

# Step 1: Create val folder
os.makedirs(val_path, exist_ok=True)

# Step 2: Get classes, ignoring .DS_Store
classes = [cls for cls in os.listdir(train_path) if os.path.isdir(os.path.join(train_path, cls))]

# Step 3: Compute split
original_train_total = 4187
original_val_total   = 1045
total_train_images   = sum(len([f for f in os.listdir(os.path.join(train_path, cls))
                                if f.lower().endswith(('.png','.jpg','.jpeg'))])
                           for cls in classes)

train_counts = {cls: round(len([f for f in os.listdir(os.path.join(train_path, cls))
                                if f.lower().endswith(('.png','.jpg','.jpeg'))]) / total_train_images * original_train_total)
                for cls in classes}

# Step 4: Move images from train → val
for cls in classes:
    cls_train_path = os.path.join(train_path, cls)
    cls_val_path   = os.path.join(val_path, cls)
    os.makedirs(cls_val_path, exist_ok=True)

    images = [f for f in os.listdir(cls_train_path) if f.lower().endswith(('.png','.jpg','.jpeg'))]
    random.shuffle(images)

    num_train = train_counts[cls]
    train_imgs = images[:num_train]
    val_imgs   = images[num_train:]

    for img in val_imgs:
        shutil.move(os.path.join(cls_train_path, img), os.path.join(cls_val_path, img))

# Step 5: Verify counts
def count_images(path):
    counts = {}
    total = 0
    for cls in os.listdir(path):
        cls_path = os.path.join(path, cls)
        if not os.path.isdir(cls_path):
            continue
        num = len([f for f in os.listdir(cls_path) if f.lower().endswith(('.png','.jpg','.jpeg'))])
        counts[cls] = num
        total += num
    return total, counts

train_total, train_class_counts = count_images(train_path)
val_total, val_class_counts     = count_images(val_path)
test_total, test_class_counts   = count_images(test_path)

print("\n=== FINAL COUNTS ===")
print(f"Train: {train_total} images | {train_class_counts}")
print(f"Val:   {val_total} images | {val_class_counts}")
print(f"Test:  {test_total} images | {test_class_counts}")


=== FINAL COUNTS ===
Train: 4187 images | {'PNEUMONIA': 3108, 'NORMAL': 1079}
Val:   1045 images | {'PNEUMONIA': 775, 'NORMAL': 270}
Test:  624 images | {'PNEUMONIA': 390, 'NORMAL': 234}


In [ ]:
# -------------------------------
# SETUP: Dataset paths and parameters
# -------------------------------
DATASET_PATH = '/content/drive/MyDrive/xray_datasets/dataset_chestxray2017/chest_xray'
TRAIN_DIR = os.path.join(DATASET_PATH, 'train')  # Training images folder
VAL_DIR   = os.path.join(DATASET_PATH, 'val')    # Validation images folder
TEST_DIR  = os.path.join(DATASET_PATH, 'test')   # Test images folder

IMG_SIZE = (224, 224)  # Resize all images to 224x224
BATCH_SIZE = 32        # Batch size for generators

# -------------------------------
# DATA AUGMENTATION
# -------------------------------
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Training data generator with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,              # Normalize pixel values to [0,1]
    rotation_range=15,           # Random rotations
    zoom_range=0.1,              # Random zoom
    brightness_range=[0.9,1.1],  # Random brightness adjustment
    horizontal_flip=True,        # Random horizontal flip
    width_shift_range=0.1,       # Horizontal translations
    height_shift_range=0.1       # Vertical translations
)

# Validation and test generator only normalize images
val_test_datagen = ImageDataGenerator(rescale=1./255)

# -------------------------------
# GENERATORS: Create iterators for model training and evaluation
# -------------------------------
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',  # Binary classification (Normal vs Pneumonia)
    shuffle=True
)

validation_generator = val_test_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False  # Do not shuffle for consistent evaluation
)

test_generator = val_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

# -------------------------------
# PRINT SAMPLE COUNTS
# -------------------------------
print(f"Train samples: {train_generator.samples}")
print(f"Val samples: {validation_generator.samples}")
print(f"Test samples: {test_generator.samples}")

Found 4187 images belonging to 2 classes.
Found 1045 images belonging to 2 classes.
Found 624 images belonging to 2 classes.
Train samples: 4187
Val samples: 1045
Test samples: 624


In [ ]:
# ==========================================
# CHEST X-RAY CLASSIFICATION - HIC DATASET
# ==========================================

# -------------------------------
# IMPORT LIBRARIES
# -------------------------------
import os
import gc
import time
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import f1_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')  # suppress warnings for cleaner output

# -------------------------------
# CONFIG & PATHS
# -------------------------------
IMG_SIZE = (224, 224)         # Resize all images to 224x224
BATCH_SIZE = 32               # Batch size for training
EPOCHS = 20                   # Max number of epochs
NUM_CLASSES = 2               # Binary classification
CLASS_NAMES = ['NORMAL', 'PNEUMONIA']  # Class labels
COUNTRY_INCOME_LEVEL = "HIC"
DATASET_NAME = "dataset_chestxray2017"

# Persistent paths on Google Drive
PERSISTENT_DATA_DIR = "/content/drive/MyDrive/xray_datasets"
RESULTS_DIR = "/content/drive/MyDrive/xray_research_results"
os.makedirs(RESULTS_DIR, exist_ok=True)  # create results folder if missing

DATASET_PATH = os.path.join(PERSISTENT_DATA_DIR, DATASET_NAME, 'chest_xray')
TRAIN_DIR = os.path.join(DATASET_PATH, 'train')
VAL_DIR   = os.path.join(DATASET_PATH, 'val')
TEST_DIR  = os.path.join(DATASET_PATH, 'test')

# Ensure all directories exist
for d in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    if not os.path.exists(d):
        raise ValueError(f"Directory not found: {d}")

print(f"Using dataset from: {DATASET_PATH}")
print(f"Train: {TRAIN_DIR}, Val: {VAL_DIR}, Test: {TEST_DIR}")

# -------------------------------
# DATA GENERATORS
# -------------------------------
# Training generator with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.1,
    brightness_range=[0.9,1.1],
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

# Validation & test generator without augmentation
val_test_datagen = ImageDataGenerator(rescale=1./255)

# Create generators
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True
)
validation_generator = val_test_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)
test_generator = val_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

print(f"Train samples: {train_generator.samples}")
print(f"Val samples: {validation_generator.samples}")
print(f"Test samples: {test_generator.samples}")

# -------------------------------
# TRANSFER MODEL FUNCTION
# -------------------------------
def create_transfer_model(base_model_name, input_shape=(224,224,3), num_classes=1):
    """
    Create a transfer learning model using a pre-trained backbone.
    Freezes the base model and adds custom dense layers for binary classification.
    """
    if base_model_name == 'MobileNetV2':
        base = tf.keras.applications.MobileNetV2(input_shape=input_shape, include_top=False, weights='imagenet')
    elif base_model_name == 'EfficientNetB0':
        base = tf.keras.applications.EfficientNetB0(input_shape=input_shape, include_top=False, weights='imagenet')
    elif base_model_name == 'ResNet50':
        base = tf.keras.applications.ResNet50(input_shape=input_shape, include_top=False, weights='imagenet')
    else:
        raise ValueError(f"Unknown model: {base_model_name}")

    base.trainable = False  # freeze pretrained layers
    inputs = keras.Input(shape=input_shape)
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='sigmoid')(x)
    model = keras.Model(inputs, outputs)
    return model

# -------------------------------
# TRAIN & EVALUATE FUNCTION
# -------------------------------
def train_and_evaluate(model, model_name, train_gen, val_gen, test_gen):
    """
    Compile, train, and evaluate a model. Saves results and checkpoints.
    """
    print(f"\n{'='*50}\nTraining {model_name}\n{'='*50}")
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    # Setup checkpoint directory
    ckpt_dir = os.path.join(RESULTS_DIR, "checkpoints", DATASET_NAME, model_name)
    os.makedirs(ckpt_dir, exist_ok=True)
    ckpt_path = os.path.join(ckpt_dir, "epoch_{epoch:02d}.weights.h5")

    checkpoint_cb = keras.callbacks.ModelCheckpoint(
        filepath=ckpt_path,
        save_weights_only=True,
        save_freq='epoch',
        verbose=1
    )

    early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

    # Resume from latest checkpoint if available
    latest_ckpt = sorted(os.listdir(ckpt_dir))[-1] if os.listdir(ckpt_dir) else None
    if latest_ckpt:
        print(f"Resuming from checkpoint: {latest_ckpt}")
        model.load_weights(os.path.join(ckpt_dir, latest_ckpt))

    # Train the model
    start_time = time.time()
    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=EPOCHS,
        callbacks=[early_stop, checkpoint_cb],
        verbose=1
    )
    training_time = (time.time() - start_time)/60  # convert to minutes

    # Evaluate test set
    test_loss, test_acc = model.evaluate(test_gen, verbose=0)
    predictions = model.predict(test_gen)
    y_pred = (predictions > 0.5).astype(int).flatten()
    y_true = test_gen.classes

    # Compute F1 scores per class & weighted
    f1_per_class = [
        f1_score(y_true==0, y_pred==0),
        f1_score(y_true==1, y_pred==1)
    ]
    f1_weighted = f1_score(y_true, y_pred, average='weighted')
    cm = confusion_matrix(y_true, y_pred)

    # Prepare results dictionary
    results = {
        'dataset_name': DATASET_NAME,
        'country_income': COUNTRY_INCOME_LEVEL,
        'model_name': model_name,
        'num_classes': NUM_CLASSES,
        'class_names': CLASS_NAMES,
        'f1_per_class': f1_per_class,
        'f1_weighted': float(f1_weighted),
        'confusion_matrix': cm.tolist(),
        'training_time_minutes': float(training_time),
        'num_images_train': train_gen.samples,
        'num_images_val': val_gen.samples,
        'num_images_test': test_gen.samples,
        'num_parameters': int(model.count_params()),
        'test_accuracy': float(test_acc),
        'epochs_trained': len(history.history['loss'])
    }

    # Save results to JSON
    result_file = os.path.join(RESULTS_DIR, f"{DATASET_NAME}_{model_name}_results.json")
    with open(result_file, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"Results saved to: {result_file}")

    # Cleanup to free memory
    del model
    tf.keras.backend.clear_session()
    gc.collect()

    return results

# -------------------------------
# TRAIN ONLY TRANSFER MODELS
# -------------------------------
models_to_train = {
    "MobileNetV2": lambda: create_transfer_model('MobileNetV2'),
    "EfficientNetB0": lambda: create_transfer_model('EfficientNetB0'),
    "ResNet50": lambda: create_transfer_model('ResNet50')
}

all_results = []
for name, build_fn in models_to_train.items():
    model = build_fn()
    all_results.append(train_and_evaluate(model, name, train_generator, validation_generator, test_generator))

# -------------------------------
# SUMMARY
# -------------------------------
print("\n" + "="*60)
print("TRAINING COMPLETE - SUMMARY")
print("="*60)
for result in all_results:
    print(f"\n{result['model_name']}:")
    print(f"  Weighted F1: {result['f1_weighted']:.4f}")
    print(f"  Test Accuracy: {result['test_accuracy']:.4f}")
    print(f"  Training Time: {result['training_time_minutes']:.2f} min")
    print(f"  Parameters: {result['num_parameters']:,}")

print(f"\nAll results saved to: {RESULTS_DIR}/")

Using dataset from: /content/drive/MyDrive/xray_datasets/dataset_chestxray2017/chest_xray
Train: /content/drive/MyDrive/xray_datasets/dataset_chestxray2017/chest_xray/train, Val: /content/drive/MyDrive/xray_datasets/dataset_chestxray2017/chest_xray/val, Test: /content/drive/MyDrive/xray_datasets/dataset_chestxray2017/chest_xray/test
Found 4187 images belonging to 2 classes.
Found 1045 images belonging to 2 classes.
Found 624 images belonging to 2 classes.
Train samples: 4187
Val samples: 1045
Test samples: 624

Training MobileNetV2
Epoch 1/20
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8272 - loss: 0.4089
Epoch 1: saving model to /content/drive/MyDrive/xray_research_results/checkpoints/dataset_chestxray2017/MobileNetV2/epoch_01.weights.h5
131/131 ━━━━━━━━━━━━━━━━━━━━ 335s 3s/step - accuracy: 0.8277 - loss: 0.4078 - val_accuracy: 0.9464 - val_loss: 0.1470
Epoch 2/20
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9281 - loss: 0.1682
Epoch 2: saving model to /content/driv